In [1]:
import pandas as pd
import joblib
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [ ]:
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

In [3]:
fake


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"
...,...,...,...,...
23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016"
23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016"
23478,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016"
23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016"


In [4]:
true

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"
...,...,...,...,...
21412,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017"
21413,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017"
21414,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017"
21415,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017"


In [5]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [6]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [7]:
fake['class']=0
true['class']=1

In [8]:
data =pd.concat([fake,true],axis=0)

In [9]:
data.sample(10)

,title,text,subject,date,class
9180,Appeals court ruling will let some Kansas vote...,(Reuters) - Thousands of Kansas residents who ...,politicsNews,"June 11, 2016",1
2928,Trump Just Posted Two Rules For This Country ...,Donald Trump literally just posted the ultimat...,News,"January 20, 2017",0
20236,"France says Venezuela talks to take place, war...",PARIS (Reuters) - Venezuela s government and o...,worldnews,"September 12, 2017",1
3222,Senators unveil road map for self-driving car ...,WASHINGTON (Reuters) - A bipartisan trio of U....,politicsNews,"June 13, 2017",1
5282,"Ohio's John Kasich, former Trump rival, visits...",WASHINGTON (Reuters) - Ohio Governor John Kasi...,politicsNews,"February 24, 2017",1
13301,GUESS WHO WE SPOTTED IN THE VIP SECTION AT A C...,You won t believe who was spotted in the speci...,politics,"Aug 9, 2016",0
19825,Top Russian and U.S. generals discuss Syria bo...,"MOSCOW (Reuters) - General Valery Gerasimov, t...",worldnews,"September 17, 2017",1
10595,OOPS! MN: JUROR In Case Against Cop Who Killed...,The only two black jurors in the Philando Cast...,politics,"Jun 17, 2017",0
2040,WATCH: McCain Scorches His Own Party’s Sham ‘...,Arizona Senator John McCain seems to be the lo...,News,"March 23, 2017",0
16566,DETROIT FREE PRESS EDITOR Calls For Gruesome M...,Read anything Stephen Henderson has written ov...,Government News,"Jun 8, 2016",0


In [10]:
data = data.drop(["title","subject","date"],axis=1)

In [11]:
data.reset_index(inplace=True)

In [12]:
data.drop(['index'],axis=1,inplace=True)

In [13]:
data.sample(5)

,text,class
18299,While many on the left continue to accuse Pres...,0
34924,BEIJING/SEOUL (Reuters) - The latest U.N. sanc...,1
12719,If average voters turned on the TV for five mi...,0
35751,DUBAI (Reuters) - Saudi Arabia on Thursday wel...,1
3436,"Earlier today, Trump appeared to tank the stoc...",0


In [14]:
import re
import string

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', ' ', text)  # remove text inside []
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)  # remove URLs
    text = re.sub(r'<.*?>', ' ', text)  # remove HTML tags
    text = re.sub(r'\W', ' ', text)  # remove non-word characters
    text = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', text)  # remove punctuation
    text = re.sub(r'\n', ' ', text)  # remove newlines
    text = re.sub(r'\w*\d\w*', ' ', text)  # remove words containing numbers
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text

In [15]:
data["text"] = data["text"].apply(clean_text)

In [16]:
x = data["text"]
y = data["class"]

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.25,random_state=42)

In [17]:
vectorizer = TfidfVectorizer()
xv_train = vectorizer.fit_transform(x_train)
xv_test = vectorizer.transform(x_test)

In [18]:
lr = LogisticRegression()
lr.fit(xv_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [19]:
prediction = lr.predict(xv_test)
lr.score(xv_test,y_test)


0.9860133630289533

In [20]:
print(classification_report(y_test,prediction))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5895
           1       0.98      0.99      0.99      5330

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [21]:
joblib.dump(vectorizer,"vectorizer.jb")
joblib.dump(lr,"lr_model.jb")

['lr_model.jb']